In [ ]:
# Import libraries
import random
import math
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from transformers import LlamaTokenizer
from datasets import load_dataset

# 1) Initialization

In [ ]:
# Define model ID and load tokenizer
model_id = "daryl149/llama-2-7b-chat-hf"
tokenizer = LlamaTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Define model configuration parameters
config = {
    "vocab_size": tokenizer.vocab_size,
    "n_layers": 1,
    "embed_dim": 2048,
    "n_heads": 32,
    "n_kv_heads": 8,
    "multiple_of": 64,
    "ffn_dim_multiplier": None,
    "norm_eps": 1e-5,
    "max_batch_size": 16,
    "max_seq_len": 64,
    "device": "cuda",
}

# Load and preprocess dataset
dataset = load_dataset("glue", "ax", split="test")
dataset = dataset.select_columns(["premise", "hypothesis"])

# Tokenize a sample of the dataset
# Input: list of batch_num strings
# Output: input_ids → (batch_num, seq_len), attention_mask → (batch_num, seq_len)
test_set = tokenizer(
    random.sample(dataset["premise"], config["max_batch_size"]),
    truncation=True,
    max_length=config["max_seq_len"],
    padding="max_length",
    return_tensors="pt",
)

# Print keys and size of the test set
print(test_set.keys())
print(test_set["input_ids"].size())

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Generating test split:   0%|          | 0/1104 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask'])
torch.Size([16, 64])


# 2) Llama Architecture

<img src='https://miro.medium.com/v2/resize:fit:1200/1*ZbnVUpK5pw5iJJeeiBa-9w.png' width=500>

[Concept Explained](https://akgeni.medium.com/llama-concepts-explained-summary-a87f0bd61964) | [Llama Github](https://github.com/facebookresearch/llama)

## 2.1 Rotary Embedding

<img src='https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTF1FTZO0ZaN6MFKvdcRLYBqRnup_IAKvJRQAG2MC0oMw&s=10' width=800>

### 2.1.1 Implementation

In [ ]:
# Precompute theta and positional frequencies for Rotary Position Embeddings (RoPE)
def precompute_theta_pos_frequencies(head_dim, seq_len, device, theta=10000.0):

    # Compute base theta values: theta_i = 10000^(-2(i-1)/head_dim) for i in [1 .. head_dim/2]
    # Output: (head_dim // 2,)
    theta_numerator = torch.arange(0, head_dim, 2).float()
    # (head_dim // 2,)
    theta = 1.0 / (theta ** (theta_numerator / head_dim)).to(device)

    # Create position index vector
    # Output: (seq_len,)
    m = torch.arange(seq_len, device=device)

    # Compute outer product of positions and theta values
    # (seq_len,) x (head_dim // 2,) → (seq_len, head_dim // 2)
    freqs = torch.outer(m, theta).float()

    # Convert to complex polar form: exp(i * freq)
    # (seq_len, head_dim // 2) complex
    freqs_complex = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_complex


# Apply rotary embeddings to query or key tensor
def apply_rotary_embeddings(x, freqs_complex, device):

    # Reshape last dimension into pairs for complex number conversion
    # (batch_num, seq_len, num_heads, head_dim) → (batch_num, seq_len, num_heads, head_dim // 2, 2)
    x = x.float().reshape(*x.shape[:-1], -1, 2)

    # View pairs as complex numbers
    # (batch_num, seq_len, num_heads, head_dim // 2, 2) → (batch_num, seq_len, num_heads, head_dim // 2) complex
    x_complex = torch.view_as_complex(x)

    # Broadcast freqs to match batch and head dimensions
    # (seq_len, head_dim // 2) → (1, seq_len, 1, head_dim // 2) complex
    freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)

    # Apply rotation via complex multiplication
    # (batch_num, seq_len, num_heads, head_dim // 2) x (1, seq_len, 1, head_dim // 2) → (batch_num, seq_len, num_heads, head_dim // 2) complex
    x_rotated = x_complex * freqs_complex

    # Convert rotated complex values back to real pairs
    # (batch_num, seq_len, num_heads, head_dim // 2) complex → (batch_num, seq_len, num_heads, head_dim // 2, 2)
    x_out = torch.view_as_real(x_rotated)

    # Flatten the last two dimensions back to head_dim
    # (batch_num, seq_len, num_heads, head_dim // 2, 2) → (batch_num, seq_len, num_heads, head_dim)
    x_out = x_out.reshape(*x.shape)

    return x_out.type_as(x).to(device)

### 2.1.2 Amendment for Huggingface Model

In [ ]:
# Patch LlamaRotaryEmbedding with NTK-Scaled RoPE to extend context window
old_init = transformers.models.llama.modeling_llama.LlamaRotaryEmbedding.__init__


# New initialization function: rescales the RoPE base frequency using NTK theory
def ntk_scaled_init(self, dim, max_position_embeddings=2048, base=10000, device=None):
    max_position_embeddings = 16384
    # alpha value controls the scaling factor
    a = 8
    # Rescale base: higher base → lower frequencies → longer effective context
    base = base * a ** (dim / (dim - 2))
    old_init(self, dim, max_position_embeddings, base, device)


# Replace the original initialization with the NTK-scaled version
transformers.models.llama.modeling_llama.LlamaRotaryEmbedding.__init__ = ntk_scaled_init

# Load tokenizer and configure 4-bit quantization via BitsAndBytes
model_path = "daryl149/llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load causal LM with 8-bit quantization and dynamic RoPE scaling
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    load_in_8bit=True,
    device_map="auto",
    rope_scaling={"type": "dynamic", "factor": 2.0},
)

In [ ]:
# Define the prompt containing the research paper text for few-shot QA
prompt = """
You are given this machine learning research paper, please read it carefully and answer the follow up question.

=== BEGIN ===
1 INTRODUCTION

Large language models (LLMs) typically come with a pre-defined context window size. For exam-
ple, inputs to LLaMA models (Touvron et al., 2023) must be fewer than 2048 tokens. This pre-set
context window limit is frequently exceeded in applications such as conducting long conversations,
summarizing long documents, or executing long-term planning. For these applications, LLMs with
longer context windows are preferred. However, training an LLM from scratch with long context
windows requires significant investments. This naturally leads to a question: Can we extend the
context window of an existing pre-trained LLM?

One straightforward approach is to fine-tune an existing pre-trained Transformer with a longer con-
text window. However, empirically, we found that models trained this way adapt to long context
windows very slowly. After training for more than 10000 batches, the effective context window
saw a minimal increase, moving from 2048 to 2560 (Table 4). This suggests that such method is
inefficient for extending to substantially longer context windows.

While certain techniques such as ALiBi (Press et al., 2022) and LeX (Sun et al., 2022) enable length
extrapolation of Transformers, i.e. train on short context windows and inference on longer ones,
many existing pre-trained LLMs, including LLaMA (Touvron et al., 2023), use positional encodings
that have weak extrapolation properties (e.g., RoPE (Su et al., 2021)). Therefore, the applicability
of these techniques for extending the context window sizes of such LLMs remains limited.

In this work, we introduce Position Interpolation to enable context window extensions for certain
existing pre-trained LLMs, including LLaMA. The key idea is, instead of extrapolation, we directly
down-scale the position indices so that the maximum position index matches the previous context
window limit in the pre-training stage. See Figure 1 for an illustration. In other words, to accom-
modate more input tokens, we interpolate the position encodings at neighboring integer positions,
utilizing the fact that position encodings can be applied on non-integer positions, as opposed to
extrapolating outside the trained positions, which may lead to catastrophic values. We verify our
approach theoretically, by showing that the interpolated attention score has a much smaller upper

bound (~ 600x smaller in LLaMA 7B setting) than the extrapolated one, and is thus much more
stable. Therefore, interpolated position encodings are easier for the model to adapt.

Empirically, we found that Position Interpolation is highly effective and efficient, requiring only a
very short period of fine-tuning for the model to fully adapt to greatly extended context windows.
We present experimental results for extending the context window to up to 32768 from the initial
2048 across 7B to 65B LLaMA models using Position Interpolation. Our results show that
=== END OF FILE ===
"""

In [ ]:
# Configure generation settings (temperature, top-k, repetition penalty)
generation_config = GenerationConfig(
    temperature=0.0,
    top_k=20,
    repetition_penalty=1.2,
)
# Print the token length of the prompt
# prompt string → tokenized: (1, seq_len); print scalar seq_len
print(
    "Prompt token length:", len(tokenizer(prompt, return_tensors="pt")["input_ids"][0])
)


# Function to print the model prediction for a given prompt
def print_predict(prompt):
    # Format the prompt with special tokens for the assistant turn
    prompt = "<|prompter|>" + prompt + "</s><|assistant|>"

    # Tokenize the prompt
    # string → input_ids → (1, seq_len)
    encoding = tokenizer(prompt, return_tensors="pt")

    # Move input ids to GPU
    # (1, seq_len) on CUDA
    input_ids = encoding["input_ids"].to("cuda")

    # Generate output tokens autoregressively
    # input_ids (1, seq_len) → result (1, seq_len + max_new_tokens)  # (inferred)
    with torch.inference_mode():
        result = model.generate(
            input_ids=input_ids,
            generation_config=generation_config,
            return_dict_in_generate=False,
            output_scores=False,
            max_new_tokens=512,
        )

    # Decode generated token ids back to text, skipping the prompt prefix
    decoded_output = tokenizer.decode(result[0][len(input_ids[0]) :])
    print(decoded_output)


# Ask the model to summarize the paper
prompt_question = (
    prompt
    + "\nPlease give me a brief summary of this research paper in a few bullet points."
)
print_predict(prompt_question)

# Ask the model to write an abstract for the paper
prompt_question = prompt + "\nPlease write me the abstract for this paper."
print_predict(prompt_question)

# Ask the model about the fine-tuning steps
prompt_question = (
    prompt
    + "\nHow many steps was the model fine tuned for the final results? Give a short answer."
)
print_predict(prompt_question)

## 2.2 RMSNorm

![](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*tjjimBzPdzuWW73y444Uzg.jpeg)

RMSNorm is extension of Layer Normalization (LayerNorm). Reason behind using RMSNorm is the computational overhead in LayerNorm. This makes improvements slow and expensive. RMSNorm achieves comparable performance against LayerNorm but reduces the running time by 7%∼64%. The key difference between LayerNorm and RMSNorm is in how the normalization factor is computed:


* Examples:
        Batch 1: (2, 4)
        Batch 2: (6, 8)

* Layer Normalization uses mean and standard deviation across all features.

        Mean: (2 + 4) / 2 = 3.0
        Standard Deviation: sqrt((1/2) * ((2 - 3)^2 + (4 - 3)^2)) = 1.0

        Mean: (6 + 8) / 2 = 7.0
        Standard Deviation: sqrt((1/2) * ((6 - 7)^2 + (8 - 7)^2)) = 1.0

        Batch 1: ((2-3)/1, (4-3)/1)
        Batch 2: ((6-7)/1, (8-7)/1)

* RMSNorm uses the root mean square of the features individually
        RMS: sqrt((2^2 + 4^2) / 2) ≈ 3.16
        RMS: sqrt((6^2 + 8^2) / 2) ≈ 7.07

        Batch 1: ( 2/3.16, 4/3.16)
        Batch 2: ( 6/7.07, 8/7.07 )

In [ ]:
# Implementation of RMSNorm (Root Mean Square Layer Normalization)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Learnable scale parameter
        # Shape: (embed_dim,)
        self.weight = nn.Parameter(torch.ones(dim))

    # Internal normalization: scale each element by inverse RMS of its row
    def _norm(self, x: torch.Tensor):
        # Compute per-row RMS and normalize
        # (batch_num, seq_len, embed_dim)
        # rsqrt: 1 / sqrt(x)
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    # Forward pass: normalize then re-scale by learned weight
    def forward(self, x: torch.Tensor):
        # Apply RMS normalization then multiply by learnable gain
        # (batch_num, seq_len, embed_dim)
        # weight (embed_dim,) broadcasts over batch_num and seq_len
        return self.weight * self._norm(x.float()).type_as(x)

## 2.3 KV Caching
![](https://miro.medium.com/v2/resize:fit:1400/1*uyuyOW1VBqmF5Gtv225XHQ.gif)

In [ ]:
# Implementation of KV Cache for efficient autoregressive inference
class KVCache:
    def __init__(self, max_batch_size, max_seq_len, n_kv_heads, head_dim, device):
        # Pre-allocate key cache tensor
        # Shape: (max_batch_size, max_seq_len, num_kv_heads, head_dim)
        self.cache_k = torch.zeros(
            (max_batch_size, max_seq_len, n_kv_heads, head_dim)
        ).to(device)
        # Pre-allocate value cache tensor
        # Shape: (max_batch_size, max_seq_len, num_kv_heads, head_dim)
        self.cache_v = torch.zeros(
            (max_batch_size, max_seq_len, n_kv_heads, head_dim)
        ).to(device)

    # Write new key/value entries at position start_pos in the cache
    # xk, xv: (batch_num, seq_len, num_kv_heads, head_dim)
    def update(self, batch_size, start_pos, xk, xv):
        self.cache_k[:batch_size, start_pos : start_pos + xk.size(1)] = xk
        self.cache_v[:batch_size, start_pos : start_pos + xv.size(1)] = xv

    # Retrieve cached keys and values up to current position
    # Output keys, values: (batch_num, start_pos + seq_len, num_kv_heads, head_dim)
    def get(self, batch_size, start_pos, seq_len):
        keys = self.cache_k[:batch_size, : start_pos + seq_len]
        values = self.cache_v[:batch_size, : start_pos + seq_len]
        return keys, values

## 2.4 Grouped Query Attention
![](https://pbs.twimg.com/media/FzjhZk5X0AYAs_-?format=jpg&name=medium)


In [ ]:
# Helper: repeat KV heads to match the number of query heads (Grouped Query Attention)
def repeat_kv(x, n_rep):
    # x: (batch_num, seq_len, num_kv_heads, head_dim)
    batch_size, seq_len, n_kv_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    else:
        # Expand KV heads by n_rep via broadcasting then reshape
        # Input:         (batch_num, seq_len, num_kv_heads, head_dim)
        # After unsqueeze:(batch_num, seq_len, num_kv_heads, 1, head_dim)
        # After expand:  (batch_num, seq_len, num_kv_heads, n_rep, head_dim)
        # Output reshape:(batch_num, seq_len, num_kv_heads * n_rep, head_dim)
        return (
            x[:, :, :, None, :]
            .expand(batch_size, seq_len, n_kv_heads, n_rep, head_dim)
            .reshape(batch_size, seq_len, n_kv_heads * n_rep, head_dim)
        )


# Implementation of Grouped Query Self-Attention (GQA) with RoPE and KV Cache
class SelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.n_heads = config["n_heads"]
        self.n_kv_heads = config["n_kv_heads"]
        self.dim = config["embed_dim"]
        self.n_kv_heads = self.n_heads if self.n_kv_heads is None else self.n_kv_heads
        self.n_heads_q = self.n_heads
        # Repetition factor: how many Q heads share each KV head
        self.n_rep = self.n_heads_q // self.n_kv_heads
        self.head_dim = self.dim // self.n_heads

        # Query projection: embed_dim → num_heads * head_dim
        self.wq = nn.Linear(self.dim, self.n_heads * self.head_dim, bias=False)
        # Key projection (GQA: fewer heads): embed_dim → num_kv_heads * head_dim
        self.wk = nn.Linear(self.dim, self.n_kv_heads * self.head_dim, bias=False)
        # Value projection (GQA: fewer heads): embed_dim → num_kv_heads * head_dim
        self.wv = nn.Linear(self.dim, self.n_kv_heads * self.head_dim, bias=False)
        # Output projection: num_heads * head_dim → embed_dim
        self.wo = nn.Linear(self.n_heads * self.head_dim, self.dim, bias=False)

        # Initialize KV Cache for inference
        self.cache = KVCache(
            max_batch_size=config["max_batch_size"],
            max_seq_len=config["max_seq_len"],
            n_kv_heads=self.n_kv_heads,
            head_dim=self.head_dim,
            device=config["device"],
        )

    # Forward pass for Grouped Query Self-Attention with RoPE and KV Cache
    def forward(self, x, start_pos, freqs_complex):

        # seq_len is 1 during autoregressive inference, full seq_len during training
        batch_size, seq_len, _ = x.shape

        # Project input to query space
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, num_heads * head_dim)
        xq = self.wq(x)

        # Project input to key space (GQA: fewer KV heads than Q heads)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, num_kv_heads * head_dim)
        xk = self.wk(x)
        # Project input to value space
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, num_kv_heads * head_dim)
        xv = self.wv(x)

        # Reshape Q into separate heads
        # (batch_num, seq_len, num_heads * head_dim) → (batch_num, seq_len, num_heads, head_dim)
        xq = xq.view(batch_size, seq_len, self.n_heads_q, self.head_dim)

        # Reshape K into separate KV heads
        # (batch_num, seq_len, num_kv_heads * head_dim) → (batch_num, seq_len, num_kv_heads, head_dim)
        xk = xk.view(batch_size, seq_len, self.n_kv_heads, self.head_dim)
        # Reshape V into separate KV heads
        # (batch_num, seq_len, num_kv_heads * head_dim) → (batch_num, seq_len, num_kv_heads, head_dim)
        xv = xv.view(batch_size, seq_len, self.n_kv_heads, self.head_dim)

        # Apply Rotary Position Embeddings (RoPE) to query vectors
        # (batch_num, seq_len, num_heads, head_dim)
        xq = apply_rotary_embeddings(xq, freqs_complex, device=x.device)

        # Apply Rotary Position Embeddings (RoPE) to key vectors
        # (batch_num, seq_len, num_kv_heads, head_dim)
        xk = apply_rotary_embeddings(xk, freqs_complex, device=x.device)

        # Write current K, V into the KV cache at start_pos
        # xk, xv: (batch_num, seq_len, num_kv_heads, head_dim) stored into cache
        self.cache.update(batch_size, start_pos, xk, xv)

        # Read all accumulated K, V from cache (covers positions 0 .. start_pos + seq_len)
        # Output keys, values: (batch_num, start_pos + seq_len, num_kv_heads, head_dim)
        keys, values = self.cache.get(batch_size, start_pos, seq_len)

        # Expand KV heads to match number of Q heads (GQA → MHA-equivalent via repetition)
        # (batch_num, start_pos + seq_len, num_kv_heads, head_dim) → (batch_num, start_pos + seq_len, num_heads, head_dim)
        keys = repeat_kv(keys, self.n_rep)
        values = repeat_kv(values, self.n_rep)

        # Transpose Q to (batch_num, num_heads, seq_len, head_dim) for batched matmul
        # (batch_num, seq_len, num_heads, head_dim) → (batch_num, num_heads, seq_len, head_dim)
        xq = xq.transpose(1, 2)

        # Transpose K to (batch_num, num_heads, start_pos + seq_len, head_dim)
        # (batch_num, start_pos + seq_len, num_heads, head_dim) → (batch_num, num_heads, start_pos + seq_len, head_dim)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention scores
        # (batch_num, num_heads, seq_len, head_dim) @ (batch_num, num_heads, head_dim, start_pos + seq_len) → (batch_num, num_heads, seq_len, start_pos + seq_len)
        scores = torch.matmul(xq, keys.transpose(2, 3)) / math.sqrt(self.head_dim)

        # Apply softmax over key-length dimension to get attention weights
        # (batch_num, num_heads, seq_len, start_pos + seq_len)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)

        # Weighted sum of value vectors
        # (batch_num, num_heads, seq_len, start_pos + seq_len) @ (batch_num, num_heads, start_pos + seq_len, head_dim) → (batch_num, num_heads, seq_len, head_dim)
        output = torch.matmul(scores, values)

        # Merge heads and flatten back to embed_dim
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, embed_dim)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        # Final output projection
        # (batch_num, seq_len, embed_dim)
        return self.wo(output)

## 2.5 SwiGlu

![](https://miro.medium.com/v2/resize:fit:596/format:webp/1*W1sg5_qhRfOCWOM1hkXi_Q.png)

In [ ]:
# Sigmoid activation function (gating component in SwiGLU)
def sigmoid(x):
    # any shape → same shape (element-wise)
    return 1 / (1 + torch.exp(-x))


# SwiGLU activation: x * sigmoid(x) — smooth non-monotonic gating used in LLaMA FFN
def swiglu(x):
    # (batch_num, seq_len, intermediate_dim)
    return x * sigmoid(x)

## 2.6 FeedForward Layer and Decoder

In [ ]:
# Implementation of the SwiGLU Feed-Forward Network (FFN)
class FeedForward(nn.Module):
    def __init__(self, config):

        super().__init__()

        # Compute intermediate_dim using LLaMA formula: 2/3 * 4 * embed_dim
        hidden_dim = 4 * config["embed_dim"]
        hidden_dim = int(2 * hidden_dim / 3)

        if config["ffn_dim_multiplier"] is not None:
            hidden_dim = int(config["ffn_dim_multiplier"] * hidden_dim)

        # Round intermediate_dim up to the nearest multiple_of for hardware efficiency
        hidden_dim = config["multiple_of"] * (
            (hidden_dim + config["multiple_of"] - 1) // config["multiple_of"]
        )

        # Gate projection (w1): embed_dim → intermediate_dim
        self.w1 = nn.Linear(config["embed_dim"], hidden_dim, bias=False)
        # Up projection (w2): embed_dim → intermediate_dim  (second SwiGLU branch)
        self.w2 = nn.Linear(config["embed_dim"], hidden_dim, bias=False)
        # Down projection (w3): intermediate_dim → embed_dim
        self.w3 = nn.Linear(hidden_dim, config["embed_dim"], bias=False)

    # Forward pass: SwiGLU(gate) * up → down-project
    def forward(self, x: torch.Tensor):
        # Gate branch: apply SwiGLU to w1 projection
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, intermediate_dim)
        swish = swiglu(self.w1(x))
        # Up branch: linear projection without activation
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, intermediate_dim)
        x_V = self.w2(x)

        # Element-wise multiply gate and up branches (SwiGLU gating mechanism)
        # (batch_num, seq_len, intermediate_dim) * (batch_num, seq_len, intermediate_dim) → (batch_num, seq_len, intermediate_dim)
        x = swish * x_V

        # Down projection back to model dimension
        # (batch_num, seq_len, intermediate_dim) → (batch_num, seq_len, embed_dim)
        return self.w3(x)

In [ ]:
# Implementation of a single LLaMA Decoder Block (pre-norm architecture)
class DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.n_heads = config["n_heads"]
        self.dim = config["embed_dim"]
        self.head_dim = self.dim // self.n_heads

        # Sub-layers: grouped query self-attention and SwiGLU feed-forward
        self.attention = SelfAttention(config)
        self.feed_forward = FeedForward(config)

        # Pre-norm RMSNorm applied before attention
        # Weight shape: (embed_dim,)
        self.attention_norm = RMSNorm(self.dim, eps=config["norm_eps"])

        # Pre-norm RMSNorm applied before feed-forward
        # Weight shape: (embed_dim,)
        self.ffn_norm = RMSNorm(self.dim, eps=config["norm_eps"])

    # Forward pass: pre-norm → attention + residual, then pre-norm → FFN + residual
    def forward(self, x, start_pos, freqs_complex):

        # Apply RMSNorm, self-attention, then add residual
        # attention_norm: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        # attention output: (batch_num, seq_len, embed_dim)
        # h = x + attn_out: (batch_num, seq_len, embed_dim)
        h = x + self.attention.forward(self.attention_norm(x), start_pos, freqs_complex)

        # Apply RMSNorm, feed-forward, then add residual
        # ffn_norm: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        # ffn output: (batch_num, seq_len, embed_dim)
        # out = h + ffn_out: (batch_num, seq_len, embed_dim)
        out = h + self.feed_forward.forward(self.ffn_norm(h))
        return out

In [ ]:
# Full LLaMA Transformer: embedding → num_layers decoder blocks → RMSNorm → LM head
class Transformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab_size = config["vocab_size"]
        self.n_layers = config["n_layers"]

        # Token embedding lookup table
        # Weight shape: (vocab_size, embed_dim)
        self.tok_embeddings = nn.Embedding(self.vocab_size, config["embed_dim"])
        self.head_dim = config["embed_dim"] // config["n_heads"]

        # Stack of num_layers decoder blocks
        self.layers = nn.ModuleList()
        for layer_id in range(config["n_layers"]):
            self.layers.append(DecoderBlock(config))

        # Final RMSNorm applied before the LM head
        # Weight shape: (embed_dim,)
        self.norm = RMSNorm(config["embed_dim"], eps=config["norm_eps"])

        # Language model head: project hidden states to vocabulary logits
        # Weight shape: (vocab_size, embed_dim)
        self.output = nn.Linear(config["embed_dim"], self.vocab_size, bias=False)

        # Precompute RoPE frequency tensor for all positions
        # Output shape: (max_seq_len * 2, head_dim // 2) complex
        self.freqs_complex = precompute_theta_pos_frequencies(
            self.head_dim, config["max_seq_len"] * 2, device=(config["device"])
        )

    # Forward pass: token ids → vocabulary logits
    def forward(self, tokens, start_pos):
        # Unpack input shape
        # tokens: (batch_num, seq_len)
        batch_size, seq_len = tokens.shape

        # Embed token ids into dense vectors
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        h = self.tok_embeddings(tokens)

        # Slice precomputed RoPE frequencies for current position range
        # Output: (seq_len, head_dim // 2) complex
        freqs_complex = self.freqs_complex[start_pos : start_pos + seq_len]

        # Pass through each decoder block sequentially
        # Per block: (batch_num, seq_len, embed_dim)
        for layer in self.layers:
            h = layer(h, start_pos, freqs_complex)

        # Apply final RMSNorm
        # (batch_num, seq_len, embed_dim)
        h = self.norm(h)

        # Project to vocabulary logits
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, vocab_size)
        output = self.output(h).float()
        return output


# Initialize Transformer model and run a forward pass on the tokenized test set
model = Transformer(config).to(config["device"])
# (batch_num, seq_len) → (batch_num, seq_len, vocab_size)
res = model.forward(test_set["input_ids"].to(config["device"]), 0)
print(res.size())

x.float().reshape(*x.shape[:-1], -1, 2) torch.Size([16, 64, 32, 32, 2])
x.float().reshape(*x.shape[:-1], -1, 2) torch.Size([16, 64, 8, 32, 2])
torch.Size([16, 64, 32000])


# 3) Generation
[Original Generation Script](https://github.com/facebookresearch/llama/blob/main/llama/generation.py)

In [ ]:
# LLaMA model class wrapping Transformer + SentencePiece tokenizer for text completion
from typing import Optional
import torch
import time
from pathlib import Path
import json
from sentencepiece import SentencePieceProcessor
from tqdm import tqdm

from model import ModelArgs, Transformer


class LLaMA:

    def __init__(
        self,
        model: Transformer,
        tokenizer: SentencePieceProcessor,
        model_args: ModelArgs,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.args = model_args

    @staticmethod
    # Build the LLaMA model from checkpoint files and tokenizer
    def build(
        checkpoints_dir, tokenizer_path, load_model, max_seq_len, max_batch_size, device
    ):
        prev_time = time.time()
        if load_model:
            # Locate and load the first available .pth checkpoint
            checkpoints = sorted(Path(checkpoints_dir).glob("*.pth"))
            assert (
                len(checkpoints) > 0
            ), f"no checkpoint files found in {checkpoints_dir}"
            ckpt_path = checkpoints[0]
            print(f'Loading checkpoint "{ckpt_path}"')
            checkpoint = torch.load(ckpt_path, map_location="cpu")
            print(f"Loaded checkpoint in {time.time() - prev_time:.2f}s")
            prev_time = time.time()

        # Read model hyperparameters from params.json
        with open(Path(checkpoints_dir) / "params.json", "r") as f:
            params = json.loads(f.read())

        model_args: ModelArgs = ModelArgs(
            max_seq_len=max_seq_len,
            max_batch_size=max_batch_size,
            device=device,
            **params,
        )

        # Load SentencePiece tokenizer and set vocab_size in model args
        tokenizer = SentencePieceProcessor()
        tokenizer.load(tokenizer_path)
        model_args.vocab_size = tokenizer.vocab_size()

        # Configure default tensor dtype based on target device
        if device == "cuda":
            torch.set_default_tensor_type(torch.cuda.HalfTensor)
        else:
            torch.set_default_tensor_type(torch.BFloat16Tensor)

        # Initialize Transformer model and move to device
        model = Transformer(model_args).to(device)

        # Load pretrained weights (rope.freqs is excluded as it is recomputed)
        if load_model:
            del checkpoint["rope.freqs"]
            model.load_state_dict(checkpoint, strict=True)
            print(f"Loaded state dict in {time.time() - prev_time:.2f}s")

        return LLaMA(model, tokenizer, model_args)

    # Perform autoregressive text completion for a batch of string prompts
    def text_completion(self, prompts, temperature=0.6, top_p=0.9, max_gen_len=None):
        if max_gen_len is None:
            max_gen_len = self.args.max_seq_len - 1

        # Encode each prompt string into integer token ids
        prompt_tokens = [
            self.tokenizer.encode(prompt, out_type=int, add_bos=True, add_eos=False)
            for prompt in prompts
        ]
        batch_size = len(prompt_tokens)
        max_prompt_len = max(len(prompt) for prompt in prompt_tokens)
        total_len = min(self.args.max_seq_len, max_gen_len + max_prompt_len)

        # Allocate token buffer padded to total_len
        # Shape: (batch_num, total_len) filled with pad_id
        pad_id = self.tokenizer.pad_id()
        tokens = torch.full(
            (batch_size, total_len), pad_id, dtype=torch.long, device=device
        )
        for k, t in enumerate(prompt_tokens):
            tokens[k, : len(t)] = torch.tensor(t, dtype=torch.long, device=device)

        # Track which sequences have produced EOS
        # Shape: (batch_num,)
        eos_reached = torch.tensor([False] * batch_size, device=device)
        # Prompt mask: True for prompt positions, False for generation positions
        # Shape: (batch_num, total_len)
        prompt_tokens_mask = tokens != pad_id
        cur_iterator = tqdm(range(1, total_len), desc="Generating tokens")

        # Autoregressive decoding loop: generate one token per step
        for cur_pos in cur_iterator:
            with torch.no_grad():
                # Forward pass with single-token slice
                # tokens[:, cur_pos-1:cur_pos] → (batch_num, 1)
                # Output: logits → (batch_num, 1, vocab_size)
                logits = self.model.forward(tokens[:, cur_pos - 1 : cur_pos], cur_pos)
            if temperature > 0:
                # Apply temperature scaling and compute softmax distribution
                # logits[:, -1] → (batch_num, vocab_size) → probs: (batch_num, vocab_size)
                probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
                # Sample token from nucleus distribution
                # (batch_num, vocab_size) → (batch_num, 1)
                next_token = self._sample_top_p(probs, top_p)
            else:
                # Greedy decoding: select highest-probability token
                # (batch_num, vocab_size) → (batch_num,)
                next_token = torch.argmax(logits[:, -1], dim=-1)

            # Flatten to (batch_num,)
            next_token = next_token.reshape(-1)
            # Preserve original prompt token if position is part of prompt, else use predicted token
            # mask (batch_num,), original (batch_num,), predicted (batch_num,) → (batch_num,)
            next_token = torch.where(
                prompt_tokens_mask[:, cur_pos], tokens[:, cur_pos], next_token
            )

            # Write selected token into the buffer at current position
            tokens[:, cur_pos] = next_token
            # Update EOS tracker: True for non-prompt positions that produced EOS
            eos_reached |= (~prompt_tokens_mask[:, cur_pos]) & (
                next_token == self.tokenizer.eos_id
            )

            # Early exit if all sequences have produced EOS
            if all(eos_reached):
                break

        # Collect output tokens and decoded text, truncating at first EOS
        out_tokens = []
        out_text = []
        for prompt_index, current_prompt_tokens in enumerate(tokens.tolist()):
            if self.tokenizer.eos_id in current_prompt_tokens:
                eos_idx = current_prompt_tokens.index(self.tokenizer.eos_id)
                current_prompt_tokens = current_prompt_tokens[:eos_idx]
            out_tokens.append(current_prompt_tokens)
            out_text.append(self.tokenizer.decode(current_prompt_tokens))
        return (out_tokens, out_text)

    # Sample the next token using nucleus (top-p) sampling
    def _sample_top_p(self, probs, p):
        # Sort probabilities in descending order
        # (batch_num, vocab_size) → probs_sort, probs_idx: (batch_num, vocab_size)
        probs_sort, probs_idx = torch.sort(probs, dim=-1, descending=True)

        # Compute cumulative probability mass
        # (batch_num, vocab_size)
        probs_sum = torch.cumsum(probs_sort, dim=-1)

        # Mask tokens outside the nucleus (shift cumsum by one to keep boundary token)
        # (batch_num, vocab_size) bool
        mask = probs_sum - probs_sort > p

        # Zero out masked (out-of-nucleus) token probabilities
        probs_sort[mask] = 0.0

        # Renormalize the nucleus probabilities to sum to 1
        # (batch_num, vocab_size)
        probs_sort.div_(probs_sort.sum(dim=-1, keepdim=True))

        # Sample one token index from the nucleus distribution
        # (batch_num, vocab_size) → (batch_num, 1)
        next_token = torch.multinomial(probs_sort, num_samples=1)

        # Map sampled index back to actual vocabulary token id
        # probs_idx (batch_num, vocab_size), next_token (batch_num, 1) → (batch_num, 1)
        next_token = torch.gather(probs_idx, -1, next_token)
        return next_token


# Main execution block: build model and run text completion
if __name__ == "__main__":
    torch.manual_seed(0)

    allow_cuda = False
    device = "cuda" if torch.cuda.is_available() and allow_cuda else "cpu"

    # Define example prompts (standard, few-shot, and zero-shot)
    prompts = [
        "Simply put, the theory of relativity states that ",
        "If Google was an Italian company founded in Milan, it would",
        # Few shot prompt
        """Translate English to French:
        sea otter => loutre de mer
        peppermint => menthe poivrée
        plush girafe => girafe peluche
        cheese =>""",
        # Zero shot prompt
        """Tell me if the following person is actually Doraemon disguised as human:
        Name: Kean
        Decision:
        """,
    ]

    # Build the LLaMA model from checkpoints and tokenizer
    model = LLaMA.build(
        checkpoints_dir="llama-2-7b/",
        tokenizer_path="tokenizer.model",
        load_model=True,
        max_seq_len=1024,
        max_batch_size=len(prompts),
        device=device,
    )

    # Run text completion and print results
    # out_tokens: list of batch_num token-id lists; out_texts: list of batch_num decoded strings
    out_tokens, out_texts = model.text_completion(prompts, max_gen_len=64)
    assert len(out_texts) == len(prompts)
    for i in range(len(out_texts)):
        print(f"{out_texts[i]}")
        print("-" * 50)

# 4) HuggingFace Llama3

In [ ]:
# Import libraries for HuggingFace Llama 3 inference
import os
import transformers
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# Define tool functions for function-calling demo with Llama 3.1-Instruct
def get_current_temperature(location: str) -> float:
    """
    Get the current temperature at a location.

    Args:
        location: The location to get the temperature for, in the format "City, Country"
    Returns:
        The current temperature at the specified location in the specified units, as a float.
    """
    return 22.0


def get_current_humidity(location: str) -> float:
    """
    Get the current humidity at a location.

    Args:
        location: The location to get the humidity for, in the format "City, Country"
    Returns:
        The current humidity at the specified location in the specified units, as a float.
    """
    return 22.0


# Set Hugging Face authentication token
os.environ["HF_TOKEN"] = ""

# Load Llama 3.1-8B-Instruct tokenizer and model in bfloat16
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, trust_remote_code=True, device_map="auto"
)
# Set model to evaluation mode (disables dropout etc.)
model = model.eval()

# Define conversation messages for the chat template
messages = [
    {"role": "system", "content": "You are a bot that responds to weather queries."},
    {
        "role": "user",
        "content": "Hey, what's the temperature and humidity in Paris right now?",
    },
]

# Apply chat template to format messages with tool definitions (returns string, not tokens)
template_input = tokenizer.apply_chat_template(
    messages,
    tools=[get_current_temperature, get_current_humidity],
    tokenize=False,
    add_generation_prompt=False,
)

# Tokenize the formatted template and move to GPU
# string → input_ids → (1, seq_len)
inputs = tokenizer(template_input, return_tensors="pt").to("cuda")

# Print the decoded input to inspect the formatted prompt
print(tokenizer.decode(inputs["input_ids"][0]))

# Run autoregressive generation
# input_ids (1, seq_len) → (1, seq_len + max_new_tokens)  # (inferred)
output = model.generate(
    **inputs,
    max_new_tokens=256,
    num_beams=1,
    do_sample=True,
    temperature=0.5,
)[0]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a bot that responds to weather queries.<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}.Do not use variables.

{
    "type": "function",
    "function": {
        "name": "get_current_temperature",
        "description": "Get the current temperature at a location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The location to get the temperature for, in the format \"City, Country\""
                }
            },
            "required":

In [ ]:
# Decode generated tokens back to text, stripping the prompt prefix
# output[len(inputs[0]):] → (generated_len,) token ids → decoded string
tokenizer.decode(output[len(inputs[0]) :], skip_special_tokens=False)

'<|start_header_id|>assistant<|end_header_id|>\n\n<|python_tag|>{"name": "get_current_temperature", "parameters": {"location": "Paris, France"}}; {"name": "get_current_humidity", "parameters": {"location": "Paris, France"}}<|eom_id|>'